In [4]:
!pip install kaggle


In [5]:
import os
import re
import kaggle
import pandas as pd
import numpy as np
import json
from pyspark.sql import functions as F

In [6]:
os.environ['KAGGLE_USERNAME'] = "kallistrateseverin"
os.environ['KAGGLE_KEY'] = "KGAT_68e13934bb524e1efed42de6a7bc30c7"

In [ ]:
#!kaggle datasets download -d cornell-university/arxiv --unzip

In [7]:
file_path = '/home/jupyter/arxiv-metadata-oai-snapshot.json'

data = []
sample_size = 30000 


cols_to_keep = ['id', 'title', 'authors', 'abstract']

with open(file_path, 'r') as f:
    for i, line in enumerate(f):
        if i >= sample_size:
            break
        doc = json.loads(line)
        data.append({key: doc[key] for key in cols_to_keep})


df_all = pd.DataFrame(data)

#to map back later for data evalluation part
df_metadata = df_all[['id', 'title', 'authors']].set_index('id')

#for the similarity analysis its enoght to take into coinsideration abstract only and id to map back later
df_abstracts = df_all[['id', 'abstract']]

print("Columns in the dataset:", df_abstracts.columns.tolist())
display(df_abstracts.head())

Columns in the dataset: ['id', 'abstract']


,id,abstract
0,0704.0001,A fully differential calculation in perturba...
1,0704.0002,"We describe a new algorithm, the $(k,\ell)$-..."
2,0704.0003,The evolution of Earth-Moon system is descri...
3,0704.0004,We show that a determinant of Stirling cycle...
4,0704.0005,In this paper we show how to compute the $\L...


In [8]:
def clean_scientific_text(text):
    if not isinstance(text, str):
        return ""
    #lowercase
    text = text.lower()
    #url,emails
    text = re.sub(r'http\S+|www\.\S+|\S+@\S+', ' ', text)
    #other scientifc specific symbols $...$ or $$...$$
    text = re.sub(r'\$.*?\$', ' ', text)
    #other latex coomnds
    text = re.sub(r'\\[a-zA-Z]+(\{.*?\})?', ' ', text)
    #programming entities
    text = re.sub(r'&(?:quot|amp|lt|gt|apos|nbsp);|&#\d+;', ' ', text)
    #html tags
    text = re.sub(r'<[^>]+>', ' ', text)
    #citations
    text = re.sub(r'\[\d+(?:,\s*\d+)*\]', ' ', text)
    #utf chars
    text = re.sub(r'[\u00A0\u1680\u2000-\u200A\u202F\u205F\u3000\uFEFF\n\r\t]', ' ', text)
    #punctuation and extra spaces
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_abstracts = df_abstracts.copy()
df_abstracts['cleaned_text'] = df_abstracts['abstract'].apply(clean_scientific_text)

df_final = df_abstracts[['id', 'cleaned_text']]

print(df_final.head())

          id                                       cleaned_text
0  0704.0001  a fully differential calculation in perturbati...
1  0704.0002  we describe a new algorithm the pebble game wi...
2  0704.0003  the evolution of earth moon system is describe...
3  0704.0004  we show that a determinant of stirling cycle n...
4  0704.0005  in this paper we show how to compute the norm ...


In [34]:
from pyspark.sql import SparkSession
import findspark
from pyspark.sql.functions import udf, size #user defined func, and return length of an array or map column
from pyspark.sql.types import ArrayType, StringType

In [35]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ArxivPrepAndSimilarity")
    .getOrCreate()
)

sc = spark.sparkContext

In [36]:
df_final = df_final.rename(columns={"cleaned_text": "clean_text"})
df_spark = spark.createDataFrame(df_final)
df_spark.printSchema()
df_spark.show(5, truncate=80)

ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-273' coro=<_async_in_context.<locals>.run_in_context() done, defined at /opt/micromamba/lib/python3.12/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-274' coro=<Kernel.shell_main() running at /opt/micromamba/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /opt/micromamba/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py:563]>
/opt/micromamba/lib/python3.12/site-packages/IPython/core/compilerop.py:86: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  return compile(source, filename, symbol, self.flags | PyCF_ONLY_AST, 1)
ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-274' coro=<Kernel.shell_main() running at /opt/micromamba/lib/python3.12/site-packages/ipykernel/kernelbase.py:597> cb=[Task.task_wakeup()]>


root
 |-- id: string (nullable = true)
 |-- clean_text: string (nullable = true)

+---------+--------------------------------------------------------------------------------+
|       id|                                                                      clean_text|
+---------+--------------------------------------------------------------------------------+
|0704.0001|a fully differential calculation in perturbative quantum chromodynamics is pr...|
|0704.0002|we describe a new algorithm the pebble game with colors and use it obtain a c...|
|0704.0003|the evolution of earth moon system is described by the dark matter field flui...|
|0704.0004|we show that a determinant of stirling cycle numbers counts unlabeled acyclic...|
|0704.0005|in this paper we show how to compute the norm 0 using the dyadic grid this re...|
+---------+--------------------------------------------------------------------------------+
only showing top 5 rows


26/06/28 02:24:05 WARN TaskSetManager: Stage 45 contains a task of very large size (5516 KiB). The maximum recommended task size is 1000 KiB.


In [37]:
num_partitions = min(1024, max(32, len(df_final) // 50))
df_spark = df_spark.repartition(num_partitions)
df_spark.rdd.getNumPartitions()
df_spark.count()

26/06/28 02:24:22 WARN TaskSetManager: Stage 46 contains a task of very large size (5516 KiB). The maximum recommended task size is 1000 KiB.
26/06/28 02:24:22 WARN TaskSetManager: Stage 47 contains a task of very large size (5516 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

30000

# Finding similar abstracts (Jaccard + MinHash + LSH)

In [38]:
#tokeinizationn df_spark clean_text to filtered_tokens
from pyspark.sql.functions import split

#split on whitespace
df_tokens = df_spark.withColumn(
    "filtered_tokens",
    split(F.col("clean_text"), "\\s+")
).select("id", "filtered_tokens")

tokens_rdd = df_tokens.rdd.map(lambda row: (row["id"], row["filtered_tokens"]))

print("Loaded tokens:", df_tokens.count())
df_tokens.show(2, truncate=60)

26/06/28 02:25:00 WARN TaskSetManager: Stage 53 contains a task of very large size (5516 KiB). The maximum recommended task size is 1000 KiB.
26/06/28 02:25:01 WARN TaskSetManager: Stage 54 contains a task of very large size (5516 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Loaded tokens: 30000


26/06/28 02:25:03 WARN TaskSetManager: Stage 60 contains a task of very large size (5516 KiB). The maximum recommended task size is 1000 KiB.


+---------+------------------------------------------------------------+
|       id|                                             filtered_tokens|
+---------+------------------------------------------------------------+
|0705.0815|[netsukuku, is, a, p2p, network, system, designed, to, ha...|
|0704.3949|[efficient, injection, of, spin, polarized, electrons, in...|
+---------+------------------------------------------------------------+
only showing top 2 rows


In [39]:
#integer index for MinHash

from pyspark.sql.functions import explode

tokens_exploded = df_tokens.select("id", explode("filtered_tokens").alias("token"))
distinct_tokens = tokens_exploded.select("token").distinct()
vocab_rdd = distinct_tokens.rdd.map(lambda row: row["token"]).zipWithIndex()
vocab_list = vocab_rdd.collect()
token_to_idx = {t: i for t, i in vocab_list}
vocab_bc = sc.broadcast(token_to_idx)
M = len(token_to_idx)
print("Vocabulary size M =", M)

26/06/28 02:25:11 WARN TaskSetManager: Stage 63 contains a task of very large size (5516 KiB). The maximum recommended task size is 1000 KiB.
[Stage 71:>                                                         (0 + 4) / 4]

Vocabulary size M = 53314


In [40]:
#MinHash: k hash functions h_i(x) = (a_i * x + b_i) mod p
import numpy as np
import random

K = 100
random.seed(42)
np.random.seed(42)
# Prime p > M
p = 2**31 - 1
hash_params = [(random.randint(1, p - 1), random.randint(0, p - 1)) for _ in range(K)]
hash_params_bc = sc.broadcast(hash_params)

def tokens_to_int_set(tokens, vocab):
    if tokens is None:
        return set()
    return set(vocab.get(t, -1) for t in tokens) - {-1}

def minhash_signature(int_set, params_list):
    sig = []
    for a, b in params_list:
        if not int_set:
            sig.append(0)
            continue
        sig.append(min((a * x + b) % p for x in int_set))
    return sig

def compute_sig(row):
    doc_id, tokens = row
    v = vocab_bc.value
    params = hash_params_bc.value
    int_set = tokens_to_int_set(tokens, v)
    sig = minhash_signature(int_set, params)
    return (doc_id, sig)

signatures_rdd = tokens_rdd.map(compute_sig)
print("Sample signature length:", len(signatures_rdd.first()[1]))

Sample signature length: 100


In [41]:
#LSH banding: spliting signature into b and rows

from itertools import combinations

R = 5
B = K // R
assert B * R == K

def band_key(band_idx, band_vec):
    h = hash((band_idx, tuple(band_vec)))
    return h % (2**32)

def emit_band_keys(row):
    doc_id, sig = row
    out = []
    for i in range(B):
        start, end = i * R, (i + 1) * R
        key = band_key(i, sig[start:end])
        out.append((key, doc_id))
    return out

band_pairs = signatures_rdd.flatMap(emit_band_keys)

def pairs_from_bucket(kv):
    key, ids_iter = kv
    ids = list(ids_iter)
    if len(ids) < 2:
        return []
    return [((id1, id2) if id1 < id2 else (id2, id1)) for id1, id2 in combinations(ids, 2)]

candidate_pairs_rdd = band_pairs.groupByKey().flatMap(pairs_from_bucket).distinct()
print("Candidate pairs count:", candidate_pairs_rdd.count())

[Stage 77:=====================================================>(598 + 2) / 600]

Candidate pairs count: 33185


In [43]:
tokens_lookup = tokens_rdd.collectAsMap()
tokens_lookup_bc = sc.broadcast(tokens_lookup)
tokens_lookup = tokens_rdd.collectAsMap()
tokens_lookup_bc = sc.broadcast(tokens_lookup)

def jaccard(tokens1, tokens2):
    if tokens1 is None:
        tokens1 = []
    if tokens2 is None:
        tokens2 = []
    A, B = set(tokens1), set(tokens2)
    inter = len(A & B)
    total = len(A) + len(B) - inter
    if total == 0:
        return 0.0
    return inter / total

def compute_jaccard(pair):
    id1, id2 = pair
    lookup = tokens_lookup_bc.value
    t1 = lookup.get(id1, [])
    t2 = lookup.get(id2, [])
    sim = jaccard(t1, t2)
    return (id1, id2, sim)

JACCARD_THRESHOLD = 0.5
similar_pairs_rdd = candidate_pairs_rdd.map(compute_jaccard).filter(lambda x: x[2] >= JACCARD_THRESHOLD)
similar_pairs = similar_pairs_rdd.collect()
print("Similar pairs above threshold:", len(similar_pairs))
if similar_pairs:
    for (id1, id2, sim) in similar_pairs[:5]:
        print(id1, id2, round(sim, 4))

def compute_jaccard(pair):
    id1, id2 = pair
    lookup = tokens_lookup_bc.value
    t1 = lookup.get(id1, [])
    t2 = lookup.get(id2, [])
    sim = jaccard(t1, t2)
    return (id1, id2, sim)

JACCARD_THRESHOLD = 0.5
similar_pairs_rdd = candidate_pairs_rdd.map(compute_jaccard).filter(lambda x: x[2] >= JACCARD_THRESHOLD)
similar_pairs = similar_pairs_rdd.collect()
print("Similar pairs above threshold:", len(similar_pairs))
if similar_pairs:
    for (id1, id2, sim) in similar_pairs[:5]:
        print(id1, id2, round(sim, 4))

Similar pairs above threshold: 669
0704.0736 0705.2294 0.6364
0710.1372 0710.2459 0.8286
0704.0156 0706.0379 1.0
0706.0379 0708.4301 1.0
0704.0156 0707.1206 0.7778


[Stage 101:====================================================>(595 + 4) / 600]

Similar pairs above threshold: 669
0704.0736 0705.2294 0.6364
0710.1372 0710.2459 0.8286
0704.0156 0706.0379 1.0
0706.0379 0708.4301 1.0
0704.0156 0707.1206 0.7778
